# BraSNAM: download dos artigos e extração dos cabeçalhos

In [4]:
%pip install -q requests beautifulsoup4 pymupdf

Note: you may need to restart the kernel to use updated packages.


## Configuração

In [ ]:
from pathlib import Path

SAIDA = Path("brasnam")          # pasta de saída
SEM_PDF = False                  # True = baixa só os metadados (Parte 2 precisa dos PDFs)
ANOS = None                      # ex.: [2012, 2013] para limitar a alguns anos; None = todos
PAUSA = 1.0                      # segundos entre requisições, para não sobrecarregar o servidor

PDFS = SAIDA / "pdfs"            # onde a Parte 2 procura os PDFs (busca em subpastas)
METADADOS = SAIDA / "metadados_sol.json"  # gerado pela Parte 1; se existir, título, ano e nomes vêm de lá

SAIDA.mkdir(exist_ok=True)

---
# Parte 1: baixar os artigos

Lê o arquivo de edições do BraSNAM na SOL, entra em cada artigo, lê as meta tags (padrão Google Scholar) e baixa o PDF.

In [ ]:
import json
import re
import time

import requests
from bs4 import BeautifulSoup

BASE = "https://sol.sbc.org.br/index.php/brasnam"

sessao = requests.Session()
sessao.headers["User-Agent"] = "Mozilla/5.0 (pesquisa academica - levantamento bibliometrico BraSNAM)"

In [ ]:
def get(url, tentativas=3):
    for i in range(tentativas):
        try:
            r = sessao.get(url, timeout=60)
            r.raise_for_status()
            time.sleep(PAUSA)
            return r
        except requests.RequestException as e:
            print(f"  erro em {url}: {e} (tentativa {i + 1}/{tentativas})")
            time.sleep(5 * (i + 1))
    return None


def listar_edicoes():
    """Retorna lista de (url_da_edicao, titulo) a partir do arquivo de edicoes."""
    edicoes, vistos, pagina = [], set(), 1
    while True:
        url = f"{BASE}/issue/archive" + ("" if pagina == 1 else f"/{pagina}")
        r = get(url)
        if r is None:
            break
        soup = BeautifulSoup(r.text, "html.parser")
        novos = 0
        for a in soup.find_all("a", href=re.compile(r"/issue/view/\d+$")):
            href = a["href"]
            if href in vistos:
                continue
            vistos.add(href)
            titulo = a.get_text(" ", strip=True)
            if titulo:
                edicoes.append((href, titulo))
                novos += 1
        if novos == 0:
            break
        pagina += 1
    return edicoes


def listar_artigos(url_edicao):
    r = get(url_edicao)
    if r is None:
        return []
    soup = BeautifulSoup(r.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=re.compile(r"/article/view/\d+$")):
        if a["href"] not in links:
            links.append(a["href"])
    return links


def ler_artigo(url_artigo):
    """Le as meta tags (padrao Google Scholar) da pagina do artigo."""
    r = get(url_artigo)
    if r is None:
        return None
    soup = BeautifulSoup(r.text, "html.parser")

    def meta(nome):
        tag = soup.find("meta", attrs={"name": nome})
        return tag["content"].strip() if tag and tag.get("content") else ""

    # autores e instituicoes aparecem intercalados, na ordem do artigo
    autores = []
    for tag in soup.find_all("meta", attrs={"name": ["citation_author", "citation_author_institution"]}):
        valor = (tag.get("content") or "").strip()
        if tag["name"] == "citation_author":
            autores.append({"autor": valor, "instituicoes": []})
        elif autores and valor:
            autores[-1]["instituicoes"].append(valor)

    pdf = meta("citation_pdf_url")
    if not pdf:
        a = soup.find("a", class_=re.compile("galley")) or soup.find("a", string=re.compile("PDF", re.I))
        pdf = a["href"] if a and a.get("href") else ""
    pdf = pdf.replace("/article/view/", "/article/download/")

    return {
        "id": re.search(r"/article/view/(\d+)", url_artigo).group(1),
        "titulo": meta("citation_title"),
        "data": meta("citation_date") or meta("citation_publication_date"),
        "doi": meta("citation_doi"),
        "url": url_artigo,
        "pdf_url": pdf,
        "autores": autores,
    }


def baixar_pdf(url, destino):
    if destino.exists() and destino.stat().st_size > 0:
        return True
    r = get(url)
    if r is None or not r.content.startswith(b"%PDF"):
        print(f"  nao consegui baixar o PDF: {url}")
        return False
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_bytes(r.content)
    return True

### Executar o download

In [ ]:
print("Listando edicoes...")
edicoes = listar_edicoes()
if not edicoes:
    raise RuntimeError("Nenhuma edicao encontrada. Verifique sua conexao ou se o endereco do site mudou.")
print(f"{len(edicoes)} edicoes encontradas.")

metadados = []
for url_ed, titulo_ed in edicoes:
    m = re.search(r"(20\d\d)", titulo_ed)
    ano = int(m.group(1)) if m else None
    if ANOS and ano not in ANOS:
        continue
    artigos = listar_artigos(url_ed)
    print(f"\n[{ano}] {titulo_ed} -> {len(artigos)} artigos")

    for i, url_art in enumerate(artigos, 1):
        art = ler_artigo(url_art)
        if art is None:
            continue
        print(f"  {i}/{len(artigos)} {art['titulo'][:70]}")

        caminho_pdf = ""
        if not SEM_PDF and art["pdf_url"]:
            destino = PDFS / str(ano) / f"{art['id']}.pdf"
            if baixar_pdf(art["pdf_url"], destino):
                caminho_pdf = str(destino)

        metadados.append({
            "id": art["id"], "ano": ano, "edicao": titulo_ed, "titulo": art["titulo"],
            "doi": art["doi"], "url": art["url"], "pdf": caminho_pdf,
            "autores": art["autores"],  # [{"autor": ..., "instituicoes": [...]}], na ordem do artigo
        })

Path(METADADOS).write_text(json.dumps(metadados, ensure_ascii=False, indent=1), encoding="utf-8")

n_autorias = sum(len(m["autores"]) for m in metadados)
sem_inst = sum(1 for m in metadados for a in m["autores"] if not a["instituicoes"])
print(f"\nPronto: {len(metadados)} artigos, {n_autorias} autorias.")
print(f"Autorias sem instituicao nos metadados do site: {sem_inst} (a Parte 2 tenta completar pelo PDF).")
print(f"Metadados em: {Path(METADADOS).resolve()}")

---
# Parte 2: montar a base de autores

Etapas:

1. Extrai o texto de cada PDF para `.txt` (`txt/<ano>/<id>.txt`).
2. Separa só o cabeçalho, até "Abstract"/"Resumo" (`cabecalhos/<ano>/<id>.txt`).
3. Identifica título, todos os autores e a afiliação de cada um: instituição, município, UF e região.
4. Se existir `metadados_sol.json` (Parte 1), usa o título, o ano e os nomes/ordem dos autores do site, que são mais confiáveis.
   A afiliação vem do PDF; quando o PDF não traz a instituição de um autor, usa a do site.
   Artigos sem PDF entram só com os dados do site.
5. Gera `artigos.csv` e `autores.csv`.

Pode rodar só esta parte se os PDFs já estiverem em `PDFS` (rode antes a célula de configuração).

In [ ]:
import csv
import difflib
import json
import re
import unicodedata
from pathlib import Path

try:
    import pymupdf
except ImportError:
    import fitz as pymupdf  # versoes antigas do PyMuPDF

### Limpeza do texto e leitura da 1ª página

In [ ]:
SOBRESCRITOS = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789")
LIGATURAS = {"ﬁ": "fi", "ﬂ": "fl", "ﬀ": "ff", "ﬃ": "ffi", "ﬄ": "ffl", "ﬅ": "st"}
ACENTOS = {"´": "́", "`": "̀", "˜": "̃", "~": "̃", "^": "̂", "ˆ": "̂", "¨": "̈"}


def limpar(texto):
    """Corrige ligaduras e acentos soltos comuns em PDFs gerados por LaTeX antigo (ex.: 'Jo˜ao' -> 'João')."""
    for k, v in LIGATURAS.items():
        texto = texto.replace(k, v)
    texto = texto.translate(SOBRESCRITOS)
    texto = re.sub(r"([´`˜~^ˆ¨])\s?([A-Za-zı])",
                   lambda m: unicodedata.normalize("NFC", m.group(2).replace("ı", "i") + ACENTOS[m.group(1)]), texto)
    texto = re.sub(r"([A-Za-zı])\s?([´`˜ˆ¨])(?=\s|$)",  # acento depois da letra
                   lambda m: unicodedata.normalize("NFC", m.group(1).replace("ı", "i") + ACENTOS[m.group(2)]), texto)
    texto = re.sub(r"([cC])\s?¸", lambda m: "ç" if m.group(1) == "c" else "Ç", texto)
    texto = texto.replace("ı", "i")
    return unicodedata.normalize("NFC", re.sub(r"[ \t]+", " ", texto)).strip()


def linhas_primeira_pagina(pagina):
    """Reconstroi as linhas da 1a pagina com tamanho de fonte, juntando sobrescritos a linha certa."""
    flags = pymupdf.TEXTFLAGS_DICT & ~getattr(pymupdf, "TEXT_PRESERVE_LIGATURES", 0)
    spans = []
    for bloco in pagina.get_text("dict", flags=flags)["blocks"]:
        for linha in bloco.get("lines", []):
            for s in linha["spans"]:
                if s["text"].strip():
                    spans.append(s)
    spans.sort(key=lambda s: (s["bbox"][1], s["bbox"][0]))

    linhas = []  # cada linha: lista de spans
    for s in spans:
        y0, y1 = s["bbox"][1], s["bbox"][3]
        destino = None
        for ln in linhas[-4:]:
            ly0 = min(x["bbox"][1] for x in ln)
            ly1 = max(x["bbox"][3] for x in ln)
            sobreposicao = min(y1, ly1) - max(y0, ly0)
            if sobreposicao > 0.5 * min(y1 - y0, ly1 - ly0):
                destino = ln
                break
        if destino is None:
            linhas.append([s])
        else:
            destino.append(s)

    resultado = []
    for ln in linhas:
        ln.sort(key=lambda s: s["bbox"][0])
        tamanho = max(s["size"] for s in ln)
        centro = max(ln, key=lambda s: s["size"])
        cy = (centro["bbox"][1] + centro["bbox"][3]) / 2
        texto, anterior = "", None
        for s in ln:
            sobrescrito = (s["flags"] & 1) or (
                s["size"] < 0.85 * tamanho and (s["bbox"][1] + s["bbox"][3]) / 2 < cy - 0.1 * tamanho)
            if anterior is not None and not sobrescrito and s["bbox"][0] - anterior["bbox"][2] > 0.15 * tamanho:
                texto += " "
            elif anterior is not None and sobrescrito and texto and not texto.endswith(" "):
                pass  # sobrescrito fica colado no nome: "Silva1"
            texto += s["text"]
            if sobrescrito:
                texto += " "
            anterior = s
        texto = limpar(texto)
        if texto:
            resultado.append({"texto": texto, "tamanho": round(tamanho, 1), "y": ln[0]["bbox"][1]})
    resultado.sort(key=lambda l: l["y"])
    return resultado

### Cabeçalho: título, autores e afiliações

In [ ]:
FIM_CABECALHO = re.compile(r"^\s*(abstract|resumo|resumen)\b", re.I)
SECAO = re.compile(r"^\s*1\.?\s+(introdu|introduction)", re.I)
EMAIL = re.compile(r"@|\{[^}]*\}\s*@?|\bemail\b|\be-mail\b", re.I)
MARCA_AFILIACAO = re.compile(r"^\s*(\d{1,2}|[*†‡])\s*(?=[A-Za-zÀ-ÿ(])(.*)$")

INST_FORTE = re.compile(
    r"\b(Universidade|Universit(y|é|ät|à|at)|Universidad|Univ\.|Instituto Federal|Centro Federal|CEFET|"
    r"Faculdades?|Centro Universit[aá]rio|College|SENAI|Embrapa|Fiocruz|Petrobras|INPE|IMPA|LNCC|RNP|"
    r"Escola Superior|Escola Polit[eé]cnica|Polit[eé]cnic|Institut[oe]? Nacional|Funda[cç][aã]o)\b", re.I)
UNIVERSIDADE = re.compile(  # tem prioridade sobre escola/instituto/departamento interno
    r"\b(Universidade|Universit(y|é|ät|à|at)|Universidad|Univ\.|Instituto Federal|Centro Federal|CEFET|"
    r"Centro Universit[aá]rio)\b", re.I)
INST_FRACA = re.compile(
    r"\b(Instituto|Institute|Institut|Centro|Center|Centre|Escola|School|Departamento|Department|Dept|Faculty|"
    r"Programa|Program|Graduate|P[oó]s-Gradua[cç][aã]o|Laborat[oó]rio|Laboratory|Lab|N[uú]cleo|Grupo|Group|"
    r"Coordena[cç][aã]o|Foundation|Empresa|Company|Inc|Ltda|Secretaria|Minist[eé]rio|Ag[eê]ncia|Agency|"
    r"Research|Pesquisa|Unidade|Divis[aã]o|Division|Reitoria)\b", re.I)


def separar_cabecalho(linhas):
    """Retorna (linhas_do_cabecalho, achou_fim). O cabecalho comeca no titulo (maior fonte)."""
    fim = None
    for i, l in enumerate(linhas):
        if FIM_CABECALHO.match(l["texto"]) or SECAO.match(l["texto"]):
            fim = i
            break
    corpo = linhas[:fim] if fim is not None else linhas[:25]
    if not corpo:
        return [], fim is not None
    maior = max(l["tamanho"] for l in corpo)
    inicio = next(i for i, l in enumerate(corpo) if l["tamanho"] >= 0.92 * maior)
    return corpo[inicio:], fim is not None


NAO_ARTIGO = re.compile(r"^(anais|apresenta[cç][aã]o|pref[aá]cio|preface|foreword|sum[aá]rio|comit[eê]s?|"
                        r"mensagem d[aoe]s?)\b", re.I)
AUTOR_COM_MARCA = re.compile(r"[A-Za-zÀ-ÿ]\d{1,2}(?:\s?,\s?\d{1,2})*\s*[,;]\s*[A-ZÀ-Ý]")  # "Silva1,2 , Ana"


def eh_nao_artigo(cab):
    """Capa dos anais, apresentacao, prefacio etc. (PDFs que estao na SOL mas nao sao artigos)."""
    return any(NAO_ARTIGO.match(l["texto"]) for l in cab[:4])


def separar_titulo(cab):
    """O titulo sao as primeiras linhas de fonte maior. Para na linha de autores ("Silva1,2 ,"),
    mesmo quando autores e titulo tem a mesma fonte."""
    maior = cab[0]["tamanho"]
    n = 1
    while n < len(cab):
        l = cab[n]
        if AUTOR_COM_MARCA.search(l["texto"]) or MARCA_AFILIACAO.match(l["texto"]) or EMAIL.search(l["texto"]):
            break
        # subtitulo um pouco menor (18pt / 16pt): so quando a linha anterior termina em ":" ou "-",
        # ou a linha comeca minuscula; senao e a linha de autores em fonte um pouco menor (16pt / 14pt)
        continua = re.search(r"[:\-–—,]$", cab[n - 1]["texto"]) or l["texto"][:1].islower()
        if l["tamanho"] >= 0.92 * maior or (l["tamanho"] >= 0.85 * maior and continua):
            n += 1
        else:
            break
    titulo = re.sub(r"-\s+(?=[a-zà-ÿ])", "", " ".join(l["texto"] for l in cab[:n]))
    titulo = re.sub(r"(?<=[a-zà-ÿ])\d$", "", titulo)  # marca de nota de rodape: "... online1"
    return titulo, cab[n:]


def eh_inicio_afiliacao(texto):
    return bool(EMAIL.search(texto) or MARCA_AFILIACAO.match(texto)
                or INST_FORTE.search(texto) or INST_FRACA.search(texto))


def juntar_linhas_autores(linhas):
    """Junta as linhas de autores. Nome quebrado no fim da linha ("... , Patrícia" / "Nicolau MAGRIS1")
    e emendado com espaco; nos outros casos a quebra de linha separa autores."""
    texto, anterior = "", ""
    for t in linhas:
        if texto:
            quebrou_nome = (re.search(r"\b(de|da|do|dos|das|del|van|von|di)$", anterior)
                            or (len(anterior) >= 70 and re.search(r"[A-Za-zÀ-ÿ.]$", anterior)))
            texto += " " if quebrou_nome else " , "
        texto += t
        anterior = t
    return texto


def parse_autores(texto):
    """'Ana Silva1,2, Bruno Souza2 e Carla Lima1' -> [('Ana Silva', ['1','2']), ...]"""
    texto = re.sub(r"\(\s*[*†‡]\s*\)", "*", texto.replace(";", ","))  # "Silva1(*) ; Souza2" -> "Silva1* , Souza2"
    texto = re.sub(r"\s+(and|e|&)\s+", ", ", texto)
    autores = []
    for seg in texto.split(","):
        seg = seg.strip(" ;")
        if not seg:
            continue
        if re.fullmatch(r"[\d\s*†‡]+", seg):
            if autores:
                autores[-1][1].extend(re.findall(r"\d+", seg))
            continue
        m = re.match(r"^(.*?)[\s\d*†‡]*$", seg)
        nome = m.group(1).strip()
        marcas = re.findall(r"\d+", seg[len(m.group(1)):])
        if nome:
            autores.append((nome, marcas))
    return autores


def parse_afiliacoes(linhas):
    """Agrupa as linhas de afiliacao por numero ('1', '2'...). Sem numeros, tudo vai para '_'."""
    blocos, atual = {}, None
    for t in linhas:
        if EMAIL.search(t):
            continue
        m = MARCA_AFILIACAO.match(t)
        if m:
            atual = m.group(1)
            blocos.setdefault(atual, []).append(m.group(2).strip())
            continue
        if atual is None:
            atual = "_"
            blocos[atual] = []
        blocos[atual].append(t)
    return blocos


SIGLA = re.compile(r"\(?(?=[A-Za-z0-9-]{3,}\)?$)[A-Z][a-z0-9-]*[A-Z][A-Za-z0-9-]*\)?")  # UFAM, UnB, PUC-Rio
PAIS_BR = re.compile(r"\b(Brasil|Brazil|BRASIL|BRAZIL)\b")


def parece_endereco(t):
    s = sem_acento(t)
    return bool(re.search(r"(?<![A-Za-z])(" + "|".join(UFS) + r")(?![A-Za-z])", t)
                or re.search(r"\d{5}-?\d{3}", t) or PAIS_BR.search(t) or EXTERIOR.search(t)
                or any(re.search(rf"\b{re.escape(c)}\b", s) for c in CIDADES_NORM))


def juntar_linhas_quebradas(linhas_bloco):
    """'Universidade Federal de Minas' + 'Gerais' -> uma linha so."""
    juntas = []
    for t in linhas_bloco:
        continuacao = (juntas and not re.search(r"\s[–—-]\s|[,;]", t) and not parece_endereco(t)
                       and not INST_FORTE.match(t) and not INST_FRACA.match(t)
                       and (INST_FORTE.search(juntas[-1]) or INST_FRACA.search(juntas[-1]))
                       and not parece_endereco(juntas[-1].split(" – ")[-1]))
        if continuacao:
            juntas[-1] = f"{juntas[-1]} {t}"
        else:
            juntas.append(t)
    return juntas


def dividir_instituicao_endereco(linhas_bloco):
    """Separa os pedacos de instituicao dos pedacos de endereco."""
    pedacos = []
    for t in juntar_linhas_quebradas(linhas_bloco):
        for p in re.split(r"\s+[–—-]\s+|\s*[,;]\s*", t):
            p = p.strip(" .")
            if p:
                pedacos.append(p)
    inst, ender = [], []
    for p in pedacos:
        nucleo = p.strip("()")
        sigla = (SIGLA.fullmatch(p) and nucleo not in UFS and not PAIS_BR.search(p)
                 and not EXTERIOR.fullmatch(nucleo))
        if INST_FORTE.search(p) or INST_FRACA.search(p) or sigla:
            inst.append(p)
        else:
            ender.append(p)
    if not inst:  # sem palavra-chave ("CPFL Energia", "Hurb.com", "CIC/UnB"): 1o pedaco que nao e endereco
        nome = next((p for p in ender if not parece_endereco(p) and not re.search(r"\d", p)
                     and not LOGRADOURO.match(p)), "")
        if nome:
            inst.append(nome)
            ender.remove(nome)
    principal = (next((p for p in inst if UNIVERSIDADE.search(p)), None)
                 or next((p for p in inst if INST_FORTE.search(p)), None) or (inst[0] if inst else ""))
    # junta a sigla que vem logo depois: "Universidade Federal do Amazonas (UFAM)"
    if principal:
        i = inst.index(principal)
        if i + 1 < len(inst) and SIGLA.fullmatch(inst[i + 1]) and "(" not in principal:
            principal = f"{principal} ({inst[i + 1].strip('()')})"
    return principal, ", ".join(ender)

### Município, UF e região

In [ ]:
UFS = {"AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG", "PA", "PB", "PR",
       "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"}
REGIAO = {**dict.fromkeys(["AC", "AP", "AM", "PA", "RO", "RR", "TO"], "Norte"),
          **dict.fromkeys(["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"], "Nordeste"),
          **dict.fromkeys(["DF", "GO", "MT", "MS"], "Centro-Oeste"),
          **dict.fromkeys(["ES", "MG", "RJ", "SP"], "Sudeste"),
          **dict.fromkeys(["PR", "RS", "SC"], "Sul")}
ESTADOS = {"acre": "AC", "alagoas": "AL", "amapa": "AP", "amazonas": "AM", "bahia": "BA", "ceara": "CE",
           "distrito federal": "DF", "espirito santo": "ES", "goias": "GO", "maranhao": "MA",
           "mato grosso do sul": "MS", "mato grosso": "MT", "minas gerais": "MG", "paraiba": "PB",
           "parana": "PR", "pernambuco": "PE", "piaui": "PI", "rio de janeiro": "RJ",
           "rio grande do norte": "RN", "rio grande do sul": "RS", "rondonia": "RO", "roraima": "RR",
           "santa catarina": "SC", "sao paulo": "SP", "sergipe": "SE", "tocantins": "TO"}
# municipios conhecidos (nome oficial -> UF); os nao listados sao achados pela posicao no endereco ("Cidade - UF")
CIDADES = {
    # capitais
    "Rio Branco": "AC", "Maceió": "AL", "Macapá": "AP", "Manaus": "AM", "Salvador": "BA", "Fortaleza": "CE",
    "Brasília": "DF", "Vitória": "ES", "Goiânia": "GO", "São Luís": "MA", "Cuiabá": "MT", "Campo Grande": "MS",
    "Belo Horizonte": "MG", "Belém": "PA", "João Pessoa": "PB", "Curitiba": "PR", "Recife": "PE",
    "Teresina": "PI", "Rio de Janeiro": "RJ", "Natal": "RN", "Porto Alegre": "RS", "Porto Velho": "RO",
    "Boa Vista": "RR", "Florianópolis": "SC", "São Paulo": "SP", "Aracaju": "SE", "Palmas": "TO",
    # Norte
    "Itacoatiara": "AM", "Parintins": "AM", "Humaitá": "AM", "Tefé": "AM", "Coari": "AM",
    "Santarém": "PA", "Marabá": "PA", "Altamira": "PA", "Castanhal": "PA", "Abaetetuba": "PA", "Tucuruí": "PA",
    "Ji-Paraná": "RO", "Araguaína": "TO", "Gurupi": "TO",
    # Nordeste
    "Arapiraca": "AL", "Delmiro Gouveia": "AL",
    "Vitória da Conquista": "BA", "Feira de Santana": "BA", "Ilhéus": "BA", "Jequié": "BA", "Camaçari": "BA",
    "Juazeiro": "BA", "Barreiras": "BA", "Alagoinhas": "BA", "Cruz das Almas": "BA", "Senhor do Bonfim": "BA",
    "Santo Antônio de Jesus": "BA",
    "Sobral": "CE", "Quixadá": "CE", "Crato": "CE", "Juazeiro do Norte": "CE", "Russas": "CE", "Crateús": "CE",
    "Iguatu": "CE", "Itapajé": "CE",
    "Imperatriz": "MA", "Codó": "MA", "Bacabal": "MA",
    "Campina Grande": "PB", "Rio Tinto": "PB", "Patos": "PB", "Cajazeiras": "PB",
    "Caruaru": "PE", "Garanhuns": "PE", "Petrolina": "PE", "Serra Talhada": "PE",
    "Parnaíba": "PI", "Picos": "PI",
    "Mossoró": "RN", "Pau dos Ferros": "RN", "Angicos": "RN", "Caicó": "RN", "Currais Novos": "RN",
    "São Cristóvão": "SE", "Itabaiana": "SE", "Lagarto": "SE",
    # Centro-Oeste
    "Taguatinga": "DF", "Planaltina": "DF", "Ceilândia": "DF",
    "Anápolis": "GO", "Rio Verde": "GO", "Jataí": "GO", "Catalão": "GO", "Luziânia": "GO",
    "Dourados": "MS", "Três Lagoas": "MS", "Corumbá": "MS", "Ponta Porã": "MS",
    "Rondonópolis": "MT", "Sinop": "MT", "Barra do Garças": "MT", "Cáceres": "MT",
    # Sudeste
    "Vila Velha": "ES", "Colatina": "ES", "Cachoeiro de Itapemirim": "ES", "São Mateus": "ES",
    "Juiz de Fora": "MG", "Uberlândia": "MG", "Viçosa": "MG", "Lavras": "MG", "Itajubá": "MG",
    "Ouro Preto": "MG", "São João del-Rei": "MG", "Montes Claros": "MG", "Divinópolis": "MG", "Uberaba": "MG",
    "Formiga": "MG", "Diamantina": "MG", "Poços de Caldas": "MG", "Varginha": "MG", "Sete Lagoas": "MG",
    "Ipatinga": "MG", "Governador Valadares": "MG", "Teófilo Otoni": "MG", "Contagem": "MG", "Betim": "MG",
    "Itabira": "MG", "João Monlevade": "MG", "Ouro Branco": "MG", "Unaí": "MG", "Bambuí": "MG",
    "Muzambinho": "MG", "Pouso Alegre": "MG", "Santa Rita do Sapucaí": "MG", "Alfenas": "MG",
    "Barbacena": "MG", "Ituiutaba": "MG", "Patos de Minas": "MG", "Araxá": "MG",
    "Niterói": "RJ", "Petrópolis": "RJ", "Seropédica": "RJ", "Nova Iguaçu": "RJ", "Volta Redonda": "RJ",
    "Macaé": "RJ", "Campos dos Goytacazes": "RJ", "Duque de Caxias": "RJ", "São Gonçalo": "RJ",
    "Cabo Frio": "RJ", "Nova Friburgo": "RJ", "Teresópolis": "RJ", "Angra dos Reis": "RJ",
    "Campinas": "SP", "São Carlos": "SP", "Santo André": "SP", "São Bernardo do Campo": "SP",
    "Ribeirão Preto": "SP", "Sorocaba": "SP", "São José dos Campos": "SP", "Guarulhos": "SP", "Bauru": "SP",
    "Presidente Prudente": "SP", "São José do Rio Preto": "SP", "Santos": "SP", "Franca": "SP",
    "Piracicaba": "SP", "Araraquara": "SP", "Rio Claro": "SP", "Botucatu": "SP", "Marília": "SP",
    "Jundiaí": "SP", "Osasco": "SP", "Barueri": "SP", "Lorena": "SP", "Guaratinguetá": "SP",
    "Ilha Solteira": "SP",
    # Sul
    "Londrina": "PR", "Maringá": "PR", "Ponta Grossa": "PR", "Cascavel": "PR", "Toledo": "PR",
    "Foz do Iguaçu": "PR", "Guarapuava": "PR", "Pato Branco": "PR", "Medianeira": "PR",
    "Cornélio Procópio": "PR", "Campo Mourão": "PR", "Apucarana": "PR", "Francisco Beltrão": "PR",
    "Dois Vizinhos": "PR",
    "Pelotas": "RS", "Santa Maria": "RS", "Alegrete": "RS", "Bagé": "RS", "Caxias do Sul": "RS",
    "São Leopoldo": "RS", "Rio Grande": "RS", "Passo Fundo": "RS", "Lajeado": "RS", "Santa Cruz do Sul": "RS",
    "Novo Hamburgo": "RS", "Canoas": "RS", "Erechim": "RS", "Ijuí": "RS", "Santo Ângelo": "RS",
    "Uruguaiana": "RS", "São Gabriel": "RS",
    "Joinville": "SC", "Blumenau": "SC", "Itajaí": "SC", "Chapecó": "SC", "Criciúma": "SC", "Tubarão": "SC",
    "Lages": "SC", "Araranguá": "SC", "Rio do Sul": "SC",
}

EXTERIOR = re.compile(
    r"\b(USA|U\.S\.A|United States|EUA|Estados Unidos|Portugal|France|Fran[cç]a|Spain|Espanha|Argentina|"
    r"Uruguay|Uruguai|Chile|Germany|Alemanha|UK|United Kingdom|Reino Unido|England|Italy|It[aá]lia|"
    r"Canada|Canad[aá]|M[eé]xico|Colombia|Col[oô]mbia|Peru|Netherlands|Holanda|Switzerland|Su[ií][cç]a|"
    r"Taiwan|China|Japan|Jap[aã]o|Australia|Austr[aá]lia|Ireland|Irlanda|Belgium|B[eé]lgica|Sweden|"
    r"Austria|India|Qatar|Israel|Finland|Norway|Denmark|Paraguay|Paraguai|Ecuador|Equador|Venezuela|"
    r"Bol[ií]via|Greece|Poland|Singapore|Korea|Coreia|Luxembourg|Luxemburgo)\b")


def sem_acento(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn").lower()


def sem_acento(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn").lower()


def norm_cidade(s):
    """Sem acento, minusculas, hifen vira espaco: 'Belo-Horizonte' e 'São João del Rei' casam com a tabela."""
    return re.sub(r"[\s-]+", " ", sem_acento(s)).strip()


# busca normalizada: "sao joao del rei" -> ("São João del-Rei", "MG")
CIDADES_NORM = {norm_cidade(nome): (nome, uf) for nome, uf in CIDADES.items()}
# nomes de estado que nao sao tambem nome de cidade: apagados antes de procurar cidade,
# para "Rio Grande do Norte" nao virar o municipio "Rio Grande"
SO_ESTADO = [n for n in sorted(ESTADOS, key=len, reverse=True) if n not in CIDADES_NORM]

UF_RE = re.compile(r"(?<![A-Za-z])(" + "|".join(sorted(UFS)) + r")(?![A-Za-z])")
UF_NO_FIM = re.compile(r"[\s/(-]+(" + "|".join(sorted(UFS)) + r")\)?$")  # "Manaus/AM", "Manaus-AM", "Manaus (AM)"
LOGRADOURO = re.compile(
    r"^(rua|r\.|av\.?|avenida|alameda|travessa|rodovia|estrada|pra[cç]a|campus|bairro|caixa postal|cx\.?|"
    r"cep|km|bloco|sala|pr[eé]dio|cidade universit[aá]ria|street|st\.|road|building|p\.?\s?o\.? box)\b", re.I)


def pecas_endereco(endereco):
    """'Av. X 123, Manaus/AM, CEP 69077-000, Brasil' -> [('Av. X 123', ''), ('Manaus', 'AM'), ('', ''), ('Brasil', '')]"""
    pecas = []
    for p in re.split(r"\s*[,;\n]\s*|\s+[–—-]\s+", endereco):
        p = re.sub(r"\bCEP\b:?|\d{5}-?\d{3}", " ", p).strip(" .")
        m = UF_NO_FIM.search(p)
        uf = ""
        if m and m.start() > 0:
            p, uf = p[:m.start()].strip(" ."), m.group(1)
        pecas.append((p.strip(" .()"), uf))
    return pecas


def parece_municipio(p):
    sigla = re.fullmatch(r"[A-Z0-9/-]{2,8}", p) or (SIGLA.fullmatch(p) and not p.isupper())  # UFRJ, PPGI, UnB
    return bool(p and p[0].isupper() and not sigla and not re.search(r"\d", p) and len(p.split()) <= 5
                and p not in UFS and not LOGRADOURO.match(p) and not PAIS_BR.search(p)
                and not EXTERIOR.search(p) and not INST_FORTE.search(p) and not INST_FRACA.search(p)
                and sem_acento(p) not in ESTADOS)


def cidade_conhecida_no_texto(texto):
    """Cidade da tabela que aparece primeiro no texto (a mais longa, se empatar)."""
    s = norm_cidade(texto)
    for estado in SO_ESTADO:
        s = re.sub(rf"\b{re.escape(estado)}\b", " ", s)
    achados = [(m.start(), -len(chave), chave) for chave in CIDADES_NORM
               for m in [re.search(rf"\b{re.escape(chave)}\b", s)] if m]
    return CIDADES_NORM[min(achados)[2]] if achados else None


def formatar_municipio(p):
    return p.title().replace(" De ", " de ").replace(" Do ", " do ").replace(" Da ", " da ").replace(
        " Dos ", " dos ").replace(" Das ", " das ") if p.isupper() else p


def achar_municipio(endereco, bloco_inteiro):
    """Retorna (municipio, uf_do_municipio). bloco_inteiro tem as linhas da afiliacao separadas por '\n'."""
    pecas = pecas_endereco(bloco_inteiro)                    # cada linha e um pedaco separado
    pecas_unidas = pecas_endereco(bloco_inteiro.replace("\n", " "))  # junta "São" + "Paulo - SP"
    # 1. pedaco que e exatamente uma cidade conhecida
    for lista in (pecas, pecas_unidas):
        for p, _ in lista:
            if norm_cidade(p) in CIDADES_NORM:
                return CIDADES_NORM[norm_cidade(p)]
    # 2. pedaco logo antes da UF, do estado ou do pais: "Cidade, UF", "Cidade/UF", "Cidade - Estado", "Lisboa, Portugal"
    for i, (p, uf) in enumerate(pecas):
        if uf and parece_municipio(p):
            return formatar_municipio(p), uf
        if i == 0 or not parece_municipio(pecas[i - 1][0]):
            continue
        anterior = formatar_municipio(pecas[i - 1][0])
        if p in UFS:
            return anterior, p
        if sem_acento(p) in ESTADOS:
            return anterior, ESTADOS[sem_acento(p)]
        if EXTERIOR.fullmatch(p):
            return anterior, ""
    # 3. cidade conhecida no meio do endereco ou, por ultimo, no nome da instituicao ("Univ. Federal de Lavras")
    for texto in (" , ".join(p for p, _ in pecas_endereco(endereco) if not LOGRADOURO.match(p)),
                  bloco_inteiro.replace("\n", " ")):
        achado = cidade_conhecida_no_texto(texto)
        if achado:
            return achado
    return "", ""


# sede de instituicoes frequentes, usada so quando o cabecalho nao traz a cidade ("UFMG - Brasil").
# Busca no texto sem acento e em minusculas; vale o primeiro padrao que casar.
# None = instituicao com varios campi espalhados (institutos federais etc.): nao deduz o municipio.
SEDES = [
    (r"\binstituto federal\b|\bfederal institute\b|\bif-?(?:[a-z]{2}|goiano|baiano|fluminense|catarinense|farroupilha|sul[a-z]*)\b", None, None),
    (r"\buemg\b|universidade do estado de minas gerais", None, None),
    (r"\butfpr\b|tecnologica federal do parana", None, None),
    (r"\bunesp\b|universidade estadual paulista", None, None),
    (r"\buesb\b|sudoeste da bahia", None, None),
    (r"\bufrrj\b|federal rural do rio de janeiro", "Seropédica", "RJ"),
    (r"\bunirio\b|federal do estado do rio de janeiro|state of rio de janeiro", "Rio de Janeiro", "RJ"),
    (r"\buerj\b|universidade do estado do rio de janeiro", "Rio de Janeiro", "RJ"),
    (r"\bpuc-?(?:rio|rj)\b|catolica do rio de janeiro|catholic university of rio de janeiro", "Rio de Janeiro", "RJ"),
    (r"\bufrj\b|\bcoppe\b|\bnce\b|federal do rio de\b|federal university of rio de\b", "Rio de Janeiro", "RJ"),
    (r"\bcefet[/ -]?rj\b|celso suckow", "Rio de Janeiro", "RJ"),
    (r"\bcefet[/ -]?mg\b|tecnologica de minas gerais", "Belo Horizonte", "MG"),
    (r"\bpuc[ -]?(?:minas|mg)\b|catolica de minas gerais", "Belo Horizonte", "MG"),
    (r"\bufmg\b|federal de minas gerais|federal university of minas gerais", "Belo Horizonte", "MG"),
    (r"\bfumec\b", "Belo Horizonte", "MG"),
    (r"\bufvjm\b|jequitinhonha", "Diamantina", "MG"),
    (r"\bufv\b|federal de vicosa|university of vicosa", "Viçosa", "MG"),
    (r"\bufop\b|federal de ouro preto|university of ouro preto", "Ouro Preto", "MG"),
    (r"\bufjf\b", "Juiz de Fora", "MG"),
    (r"\bufu\b", "Uberlândia", "MG"),
    (r"\bufsj\b", "São João del-Rei", "MG"),
    (r"\bmackenzie\b", "São Paulo", "SP"),
    (r"\bufabc\b|federal do abc", "Santo André", "SP"),
    (r"\bunicamp\b|university of campinas|estadual de campinas", "Campinas", "SP"),
    (r"\bunifesp\b|federal de sao paulo", "São Paulo", "SP"),
    (r"\bufscar\b|federal de sao carlos", "São Carlos", "SP"),
    (r"\busp\b|\beach\b|universidade de sao paulo|university of sao paulo", "São Paulo", "SP"),
    (r"\bita\b|tecnologico de aeronautica", "São José dos Campos", "SP"),
    (r"\bcpqd\b", "Campinas", "SP"),
    (r"\blncc\b|nacional de computacao cientifica", "Petrópolis", "RJ"),
    (r"\bime\b|instituto militar de engenharia", "Rio de Janeiro", "RJ"),
    (r"\bfiocruz\b|oswaldo cruz", "Rio de Janeiro", "RJ"),
    (r"\bits rio\b|instituto de tecnologia e sociedade", "Rio de Janeiro", "RJ"),
    (r"\buff\b|universidade federal fluminense|fluminense federal university", "Niterói", "RJ"),
    (r"\bufpb\b|federal d[ao] paraiba|university of paraiba", "João Pessoa", "PB"),
    (r"\bufcg\b", "Campina Grande", "PB"),
    (r"\bufrpe\b|federal rural de pernambuco", "Recife", "PE"),
    (r"\bufpe\b|federal de pernambuco|federal university of pernambuco", "Recife", "PE"),
    (r"\bupe\b|universidade de pernambuco|university of pernambuco", "Recife", "PE"),
    (r"\bufpi\b|federal do piaui|university of piaui", "Teresina", "PI"),
    (r"\bufam\b|federal do amazonas|university of amazonas", "Manaus", "AM"),
    (r"\buea\b|universidade do estado do amazonas", "Manaus", "AM"),
    (r"\bunb\b", "Brasília", "DF"),
    (r"\baeb\b|agencia espacial brasileira", "Brasília", "DF"),
    (r"\bunifor\b", "Fortaleza", "CE"),
    (r"\bufc\b|federal do ceara|university of ceara", "Fortaleza", "CE"),
    (r"\bcimatec\b", "Salvador", "BA"),
    (r"\buneb\b|universidade do estado da bahia", "Salvador", "BA"),
    (r"\bufba\b|federal da bahia|university of bahia", "Salvador", "BA"),
    (r"\buefs\b", "Feira de Santana", "BA"),
    (r"\bufal\b|federal de alagoas|university of alagoas", "Maceió", "AL"),
    (r"\bufrgs\b|federal do rio grande do sul", "Porto Alegre", "RS"),
    (r"\bufsm\b", "Santa Maria", "RS"),
    (r"\buema\b|estadual do maranhao", "São Luís", "MA"),
    (r"\bufma\b|federal do maranhao", "São Luís", "MA"),
    (r"\bufpr\b|federal do parana|university of parana", "Curitiba", "PR"),
    (r"\bufopa\b|oeste do para", "Santarém", "PA"),
    (r"\bufra\b|rural da amazonia", "Belém", "PA"),
    (r"\bufpa\b|federal do para\b|university of para\b", "Belém", "PA"),
    (r"\bufg\b|federal de goias", "Goiânia", "GO"),
    (r"\bufms\b|federal de mato grosso do sul", "Campo Grande", "MS"),
    (r"\bufmt\b|federal de mato grosso\b", "Cuiabá", "MT"),
    (r"\buft\b|federal do tocantins", "Palmas", "TO"),
    (r"\bufes\b|federal do espirito santo", "Vitória", "ES"),
    (r"\bufsc\b|federal de santa catarina", "Florianópolis", "SC"),
    (r"\bufrn\b|federal do rio grande do norte", "Natal", "RN"),
    (r"\bufs\b|federal de sergipe", "São Cristóvão", "SE"),
]


def sede_da_instituicao(texto):
    """Retorna (municipio, uf) da sede, ou ('', '') se a instituicao nao esta na tabela ou tem varios campi."""
    s = sem_acento(texto)
    for padrao, municipio, uf in SEDES:
        if re.search(padrao, s):
            return (municipio or "", uf or "")
    return "", ""


def descobrir_local(endereco, bloco_inteiro):
    """Retorna (municipio, uf, regiao). Instituicao estrangeira: regiao 'Exterior' e UF vazia."""
    municipio, uf_cidade = achar_municipio(endereco, bloco_inteiro)
    if EXTERIOR.search(bloco_inteiro) and not re.search(r"\b(Brasil|Brazil)\b", bloco_inteiro, re.I):
        return ("" if uf_cidade else municipio), "", "Exterior"
    m = UF_RE.search(endereco)
    if m:
        uf = m.group(1)
    elif uf_cidade:
        uf = uf_cidade
    elif re.search(r"\bPará\b|\b(do|of) Para\b", bloco_inteiro):  # "Pará" so com acento: "para" e preposicao
        uf = "PA"
    else:
        s = sem_acento(f"{endereco} {bloco_inteiro}")
        uf = next((ESTADOS[n] for n in sorted(ESTADOS, key=len, reverse=True) if re.search(rf"\b{n}\b", s)), "")
    if not municipio:  # cabecalho sem cidade: usa a sede da instituicao, se nao contradiz a UF achada
        sede, uf_sede = sede_da_instituicao(bloco_inteiro)
        if sede and uf in ("", uf_sede):
            municipio, uf = sede, uf_sede
    return municipio, uf, REGIAO.get(uf, "")

### Processamento de um PDF e junção com os metadados do site

In [ ]:
VAZIO = {"instituicao": "", "municipio": "", "uf": "", "regiao": ""}


def local_de(linhas_bloco):
    """Linhas de uma afiliacao -> {instituicao, municipio, uf, regiao}."""
    inst, ender = dividir_instituicao_endereco(linhas_bloco)
    municipio, uf, regiao = descobrir_local(ender, "\n".join(linhas_bloco))
    return {"instituicao": inst, "municipio": municipio, "uf": uf, "regiao": regiao}


def ler_pdf(pdf, pasta_txt, pasta_cab, ano):
    """Retorna (titulo, autores, avisos). Cada autor: {nome, instituicao, municipio, uf, regiao}.
    autores = None quando o PDF nao e artigo (capa dos anais, apresentacao)."""
    doc = pymupdf.open(pdf)
    texto_completo = "\n\n".join(limpar(p.get_text()) for p in doc)
    (pasta_txt / ano).mkdir(parents=True, exist_ok=True)
    (pasta_txt / ano / f"{pdf.stem}.txt").write_text(texto_completo, encoding="utf-8")

    linhas = linhas_primeira_pagina(doc[0]) if len(doc) else []
    doc.close()
    cab, achou_fim = separar_cabecalho(linhas)
    (pasta_cab / ano).mkdir(parents=True, exist_ok=True)
    (pasta_cab / ano / f"{pdf.stem}.txt").write_text("\n".join(l["texto"] for l in cab), encoding="utf-8")

    avisos = [] if achou_fim else ["nao achei Abstract/Resumo"]
    if not cab:
        return "", [], ["cabecalho vazio (PDF escaneado?)"]
    if eh_nao_artigo(cab):
        return cab[0]["texto"], None, []  # None = nao e artigo

    titulo, resto = separar_titulo(cab)
    textos = [l["texto"] for l in resto]
    i = 0
    while i < len(textos) and not eh_inicio_afiliacao(textos[i]):
        i += 1
    autores = parse_autores(juntar_linhas_autores(textos[:i]))
    blocos = parse_afiliacoes(textos[i:])
    if len(blocos) > 1 and any(not marcas for _, marcas in autores):
        avisos.append("autor sem numero de afiliacao")

    resultado = []
    for nome, marcas in autores:
        chave = next((m for m in marcas if m in blocos), None)
        if chave is None:
            chave = "_" if "_" in blocos else next(iter(blocos), None)
        resultado.append({"nome": nome, **(local_de(blocos[chave]) if chave else VAZIO)})
    return titulo, resultado, avisos


def parecido(a, b):
    return difflib.SequenceMatcher(None, sem_acento(a), sem_acento(b)).ratio()


def montar_autores(autores_pdf, autores_site):
    """Nomes e ordem do site (quando existem), afiliacao do PDF; sem afiliacao no PDF, usa a do site."""
    if not autores_site:
        return autores_pdf
    final = []
    for k, a in enumerate(autores_site):
        par = max(autores_pdf, key=lambda p: parecido(p["nome"], a["autor"]), default=None)
        if par is None or parecido(par["nome"], a["autor"]) < 0.6:
            # nome muito diferente (abreviado, por ex.): casa pela posicao se a contagem bate
            par = autores_pdf[k] if len(autores_pdf) == len(autores_site) else None
        dados = dict(VAZIO) if par is None else {c: par[c] for c in VAZIO}
        if not dados["instituicao"] and a["instituicoes"]:
            dados = local_de(a["instituicoes"])
        final.append({"nome": a["autor"], **dados})
    return final


def posicao(ordem, total):
    if total == 1:
        return "unico"
    return "primeiro" if ordem == 1 else "ultimo" if ordem == total else "meio"

### Executar e gerar `artigos.csv` e `autores.csv`

In [ ]:
meta = {}
if Path(METADADOS).exists():
    meta = {str(m["id"]): m for m in json.loads(Path(METADADOS).read_text(encoding="utf-8"))}
    print(f"Metadados do site: {len(meta)} artigos")

pdfs = sorted(Path(PDFS).rglob("*.pdf"))
if not pdfs and not meta:
    raise FileNotFoundError(f"Nenhum PDF em {PDFS} e nenhum metadado em {METADADOS}")

linhas_artigos, linhas_autores, pendencias, ignorados = [], [], {}, []


def registrar(id_artigo, titulo, ano, autores, avisos):
    linhas_artigos.append({"id_artigo": id_artigo, "titulo": titulo, "ano": ano})
    for ordem, a in enumerate(autores, 1):
        linhas_autores.append({
            "id_artigo": id_artigo, "ordem": ordem, "posicao": posicao(ordem, len(autores)),
            "autor": a["nome"], "instituicao": a["instituicao"], "regiao": a["regiao"],
            "municipio": a["municipio"], "uf": a["uf"],
        })
        if not a["instituicao"]:
            avisos.append(f"autor {ordem} sem instituicao")
        elif a["regiao"] != "Exterior" and not (a["uf"] and a["municipio"]):
            avisos.append(f"autor {ordem} sem municipio/UF")
    if not autores:
        avisos.append("nenhum autor encontrado")
    if avisos:
        pendencias[id_artigo] = avisos


processados = set()
for n, pdf in enumerate(pdfs, 1):
    id_artigo, m = pdf.stem, meta.get(pdf.stem, {})
    # ano: metadados do site ou subpasta pdfs/<ano>/
    ano = m.get("ano") or (pdf.parent.name if pdf.parent.name.isdigit() else "")
    try:
        titulo, autores_pdf, avisos = ler_pdf(pdf, SAIDA / "txt", SAIDA / "cabecalhos", str(ano or "sem_ano"))
    except Exception as e:  # PDF corrompido etc.
        titulo, autores_pdf, avisos = "", [], [f"erro: {e}"]
    processados.add(id_artigo)
    if autores_pdf is None and not m.get("autores"):
        ignorados.append(f"{id_artigo} ({titulo[:60]})")
        print(f"{n}/{len(pdfs)} {pdf.name}  [ignorado: nao e artigo]")
        continue
    registrar(id_artigo, m.get("titulo") or titulo, ano, montar_autores(autores_pdf or [], m.get("autores", [])), avisos)
    print(f"{n}/{len(pdfs)} {pdf.name}" + (f"  [verificar: {'; '.join(pendencias[id_artigo])}]"
                                           if id_artigo in pendencias else ""))

# artigos que estao nos metadados mas nao tem PDF: entram so com os dados do site
for id_artigo, m in meta.items():
    if id_artigo not in processados:
        registrar(id_artigo, m["titulo"], m["ano"], montar_autores([], m["autores"]), ["sem PDF"])


def chave_artigo(linha):
    id_ = str(linha["id_artigo"])
    return (int(id_) if id_.isdigit() else 0, id_)


def salvar(nome, linhas, colunas):
    with open(SAIDA / nome, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=colunas)
        w.writeheader()
        for linha in linhas:
            # campo vazio de verdade: sem espacos, "None", "nan" etc.
            limpa = {}
            for c in colunas:
                v = "" if linha.get(c) is None else str(linha[c]).strip()
                limpa[c] = "" if v.lower() in ("none", "nan", "null", "n/a") else v
            w.writerow(limpa)


linhas_artigos.sort(key=chave_artigo)
linhas_autores.sort(key=lambda l: (chave_artigo(l), l["ordem"]))
salvar("artigos.csv", linhas_artigos, ["id_artigo", "titulo", "ano"])
salvar("autores.csv", linhas_autores,
       ["id_artigo", "ordem", "posicao", "autor", "instituicao", "regiao", "municipio", "uf"])

print(f"\nPronto: {len(linhas_artigos)} artigos em {SAIDA / 'artigos.csv'}")
print(f"        {len(linhas_autores)} autorias em {SAIDA / 'autores.csv'}")
if ignorados:
    print(f"Ignorados (nao sao artigos): {', '.join(ignorados)}")
print(f"{len(pendencias)} artigos para conferir (cabecalho em {SAIDA / 'cabecalhos'}):")
for id_artigo, avisos in pendencias.items():
    print(f"  {id_artigo}: {'; '.join(avisos)}")